In [ ]:
from pathlib import Path
import random

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, log_loss

In [ ]:
market = pd.read_csv("marketdata_to_events.csv")

required = {"Date", "Open", "High", "Low", "Close", "Volume", "window", 
            "event_id", "symbol", "earnings_timestamp"}
assert required.issubset(market.columns)

market["Date"] = pd.to_datetime(market["Date"])
market["earnings_timestamp"] = pd.to_datetime(market["earnings_timestamp"])
market["event_id"] = market["event_id"].astype("int64")

assert not market.duplicated(["event_id", "Date"]).any()
market = market.sort_values(["event_id", "Date"])

files = {"train": "data/training_events.csv",
         "val": "data/validation_events.csv",
         "test": "data/testing_events.csv"}

splits = {}

for name, filename in files.items():
    frame = pd.read_csv(filename)
    frame["date"] = pd.to_datetime(frame["date"])
    frame["event_id"] = frame["event_id"].astype("int64")

    assert not frame["event_id"].duplicated().any()

    splits[name] = frame.sort_values(["date", "event_id"]).reset_index(drop=True)

all_ids = pd.concat([frame["event_id"] for frame in splits.values()], ignore_index=True)

assert not all_ids.duplicated().any()
wanted_ids = set(all_ids)

print("Market rows:", len(market))
print({name: len(frame) for name, frame in splits.items()})

In [ ]:
daily_cols = ["close_return", "overnight_gap", "intraday_return", "high_low_range", 
              "close_location", "relative_volume", "volume_change"]
SEED = 42
HORIZON = 3
PRE_DAYS = 30

def build_daily_features(event):
    event = event.sort_values("Date")

    pre = event.loc[event["window"] == "pre"].tail(PRE_DAYS).copy()
    post = event.loc[event["window"] == "post"].copy()

    if len(pre) != PRE_DAYS or len(post) < HORIZON:
        raise ValueError("Insufficient trading sessions")

    prices = pre[["Open", "High", "Low", "Close"]].to_numpy()
    volume = pre["Volume"].to_numpy()

    if not np.isfinite(prices).all() or (prices <= 0).any():
        raise ValueError("Invalid prices")

    if not np.isfinite(volume).all() or (volume < 0).any():
        raise ValueError("Invalid volume")

    if (pre["High"] < pre["Low"]).any():
        raise ValueError("High below Low")

    previous_close = pre["Close"].shift(1)

    # Each reference uses only earlier sessions.
    previous_volume_mean = (
        pre["Volume"]
        .shift(1)
        .rolling(20, min_periods=1)
        .mean()
    )

    daily = pd.DataFrame(index=pre.index)

    daily["close_return"] = np.log(
        pre["Close"] / previous_close
    )

    daily["overnight_gap"] = np.log(
        pre["Open"] / previous_close
    )

    daily["intraday_return"] = np.log(
        pre["Close"] / pre["Open"]
    )

    daily["high_low_range"] = np.log(
        pre["High"] / pre["Low"]
    )

    price_range = pre["High"] - pre["Low"]

    daily["close_location"] = (
        (pre["Close"] - pre["Low"])
        / price_range.replace(0, np.nan)
    ).fillna(0.5)

    daily["relative_volume"] = np.log(
        (pre["Volume"] + 1) / (previous_volume_mean + 1)
    )

    daily["volume_change"] = np.log(
        (pre["Volume"] + 1)
        / (pre["Volume"].shift(1) + 1)
    )

    # First row has no previous close or previous volume.
    daily = daily.iloc[1:].copy()
    sequence = daily[daily_cols].to_numpy(dtype=np.float32)

    assert sequence.shape == (PRE_DAYS - 1, len(daily_cols))
    assert np.isfinite(sequence).all()

    features = {}
    closes = pre["Close"].to_numpy()

    for window in [5, 10, 20, PRE_DAYS - 1]:
        tail = daily.tail(window)

        features[f"momentum_{window}"] = np.expm1(
            tail["close_return"].sum()
        )

        features[f"volatility_{window}"] = (
            tail["close_return"].std()
        )

        features[f"range_mean_{window}"] = (
            tail["high_low_range"].mean()
        )

        features[f"relative_volume_mean_{window}"] = (
            tail["relative_volume"].mean()
        )

        features[f"overnight_mean_{window}"] = (
            tail["overnight_gap"].mean()
        )

        features[f"intraday_mean_{window}"] = (
            tail["intraday_return"].mean()
        )

        path = closes[-(window + 1):]
        features[f"max_drawdown_{window}"] = (
            path / np.maximum.accumulate(path) - 1
        ).min()

    features["volatility_ratio_5_20"] = (
        features["volatility_5"]
        / (features["volatility_20"] + 1e-8)
    )

    features["volume_ratio_5_20"] = (
        (pre["Volume"].tail(5).mean() + 1)
        / (pre["Volume"].tail(20).mean() + 1)
    )

    assert np.isfinite(list(features.values())).all()

    # Post-event rows are used only for target checks and purging.
    target_close = float(post["Close"].iloc[HORIZON - 1])

    if not np.isfinite(target_close) or target_close <= 0:
        raise ValueError("Invalid target close")

    expected_return = (
        target_close / float(pre["Close"].iloc[-1]) - 1
    )

    metadata = {
        "event_id": int(event["event_id"].iloc[0]),
        "market_symbol": event["symbol"].iloc[0],
        "market_timestamp": event["earnings_timestamp"].iloc[0],
        "label_end": post["Date"].iloc[HORIZON - 1],
        "expected_return": expected_return
    }

    return sequence, features, metadata


sequences = {}
feature_rows = []

selected_market = market.loc[market["event_id"].isin(wanted_ids)]

for event_id, event in selected_market.groupby("event_id"):
    sequence, features, metadata = build_daily_features(event)
    sequences[int(event_id)] = sequence
    feature_rows.append({**metadata, **features})


feature_table = pd.DataFrame(feature_rows)

metadata_cols = {
    "event_id", "market_symbol", "market_timestamp",
    "label_end", "expected_return"
}

FEATURE_COLS = [col for col in feature_table.columns if col not in metadata_cols]

print("Usable events:", len(sequences))
print("Engineered features:", len(FEATURE_COLS))

feature_table.head()

In [ ]:
feature_table.columns

In [ ]:
sequence.shape

## Build CNN inputs from sequences

In [ ]:
SEED = 42
LABEL_COL = "label_3d"

BATCH_SIZE = 128
MAX_EPOCHS = 40
PATIENCE = 6
LEARNING_RATE = 1e-3
EMBEDDING_DIM = 32
DROPOUT = 0.25
N_COMPONENTS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [ ]:
raw_sequences = {}
X_sequence = {}
labels = {}
event_ids = {}

for name, filename in files.items():
    frame = pd.read_csv(filename)
    event_ids[name] = frame["event_id"].to_numpy(dtype=np.int64)

    # Match the original split's row order.
    raw_sequences[name] = np.stack([sequences[int(event_id)] for event_id in event_ids[name]]).astype(np.float32)
    labels[name] = frame[LABEL_COL].to_numpy(dtype=np.float32)

    assert np.isfinite(raw_sequences[name]).all()

sequence_length = raw_sequences["train"].shape[1]
n_channels = raw_sequences["train"].shape[2]

for name in files:
    assert raw_sequences[name].shape[1:] == (sequence_length, n_channels)

# Fit channel scaling on training data only.
sequence_scaler = StandardScaler().fit(raw_sequences["train"].reshape(-1, n_channels))

for name, raw in raw_sequences.items():
    scaled = sequence_scaler.transform(raw.reshape(-1, n_channels)).reshape(raw.shape)
    X_sequence[name] = np.ascontiguousarray(scaled.transpose(0, 2, 1), dtype=np.float32)
    print(name, X_sequence[name].shape)


def make_sequence_loader(name, shuffle=False):
    dataset = TensorDataset(torch.from_numpy(X_sequence[name]), torch.from_numpy(labels[name]))

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED)
    )


loaders = {
    name: make_sequence_loader(name, shuffle=(name == "train"))
    for name in files
}

joblib.dump(sequence_scaler, "data/cnn/sequence_scaler.joblib")

In [ ]:
class DailySequenceCNN(nn.Module):
    def __init__(self, n_channels, embedding_dim=32, dropout=0.25):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(
                n_channels, 32,
                kernel_size=3, padding=1
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                32, 32,
                kernel_size=3, padding=2, dilation=2
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                32, embedding_dim,
                kernel_size=3, padding=4, dilation=4
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )

        # Used only to train the CNN.
        self.classifier = nn.Linear(embedding_dim, 1)

    def forward(self, daily):
        embedding = self.encoder(daily)
        return self.classifier(embedding).squeeze(-1)


cnn = DailySequenceCNN(
    n_channels=n_channels,
    embedding_dim=EMBEDDING_DIM,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    cnn.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-3
)

print(cnn)